# Análisis de Contaminación en Ciudades Asiáticas Industrializadas
## Sesión 5: Caso Grupal - Tratamiento de Datos Faltantes

**Fecha**: Mayo 2026

---

## Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("Librerías importadas exitosamente")

---

## PREGUNTA 1: Revisión de Datos
### 1a. Dimensión de los datos, 1b. Data types, 1c. Últimas 12 filas

In [ ]:
# Cargar los datos
df = pd.read_excel('Sesion 5- Niveles de contaminacion ciudades asiaticas.xlsx')

# 1a. Dimensión de los datos
print("="*80)
print("1a. DIMENSIÓN DE LOS DATOS")
print("="*80)
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")
print(f"Dimensión total: {df.shape}")
print()

# 1b. Data types de cada variable
print("="*80)
print("1b. DATA TYPES DE CADA VARIABLE")
print("="*80)
print(df.dtypes)
print()

# 1c. Últimas 12 filas
print("="*80)
print("1c. ÚLTIMAS 12 FILAS DE OBSERVACIONES")
print("="*80)
df.tail(12)

### 1d. Cinco ciudades más contaminantes en términos de PM2.5

In [ ]:
print("="*80)
print("1d. CINCO CIUDADES MÁS CONTAMINANTES (PM2.5)")
print("="*80)

# Ordenar por PM2.5 descendente y obtener top 5
top_5_pm25 = df.nlargest(5, 'PM2.5')[['Ciudad', 'País', 'PM2.5']]
print(top_5_pm25.to_string())
print()

# Visualización
plt.figure(figsize=(10, 6))
top_5_data = df.nlargest(5, 'PM2.5')
plt.barh(top_5_data['Ciudad'], top_5_data['PM2.5'], color='crimson', alpha=0.7)
plt.xlabel('Concentración PM2.5 (μg/m³)', fontsize=12, fontweight='bold')
plt.ylabel('Ciudad', fontsize=12, fontweight='bold')
plt.title('Top 5 Ciudades Más Contaminadas por PM2.5', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
for i, v in enumerate(top_5_data['PM2.5']):
    plt.text(v + 2, i, str(round(v, 2)), va='center', fontweight='bold')
plt.tight_layout()
plt.show()


### 1e. Conteo de datos faltantes

In [ ]:
print("="*80)
print("1e. CONTEO DE DATOS FALTANTES")
print("="*80)

# Conteo de valores faltantes por columna
missing_count = df.isnull().sum()
missing_df = pd.DataFrame({
    'Variable': missing_count.index,
    'Valores_Faltantes': missing_count.values
})

print(missing_df.to_string(index=False))
print()
print(f"Total de valores faltantes en el dataset: {df.isnull().sum().sum()}")
print(f"Total de celdas en el dataset: {df.shape[0] * df.shape[1]}")

---

## PREGUNTA 2: Nivel de Faltabilidad de Cada Variable

In [ ]:
print("="*80)
print("PREGUNTA 2: NIVEL DE FALTABILIDAD (% DE DATOS FALTANTES)")
print("="*80)

# Calcular porcentaje de faltabilidad
faltabilidad = pd.DataFrame({
    'Variable': df.columns,
    'Valores_Faltantes': df.isnull().sum().values,
    'Total_Observaciones': len(df),
    'Porcentaje_Faltante': (df.isnull().sum().values / len(df) * 100).round(2)
})

# Ordenar por porcentaje faltante
faltabilidad = faltabilidad.sort_values('Porcentaje_Faltante', ascending=False)
print(faltabilidad.to_string(index=False))
print()

# Visualización de faltabilidad
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x <= 25 else 'orange' if x <= 50 else 'red' for x in faltabilidad['Porcentaje_Faltante']]
bars = ax.bar(faltabilidad['Variable'], faltabilidad['Porcentaje_Faltante'], color=colors, alpha=0.7)
ax.axhline(y=25, color='red', linestyle='--', linewidth=2, label='Umbral de tolerancia (25%)')
ax.set_ylabel('Porcentaje de Datos Faltantes (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Variables', fontsize=12, fontweight='bold')
ax.set_title('Faltabilidad de Datos por Variable', fontsize=14, fontweight='bold')
ax.legend()
plt.xticks(rotation=45, ha='right')
for i, (var, val) in enumerate(zip(faltabilidad['Variable'], faltabilidad['Porcentaje_Faltante'])):
    ax.text(i, val + 1, f'{val:.2f}%', ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 INTERPRETACIÓN:")
print("Variables con faltabilidad ≤ 25% (TOLERABLES): Se pueden imputar")
print("Variables con faltabilidad > 25% (NO TOLERABLES): Considerar eliminar")

---

## PREGUNTA 3: Estrategia de Tratamiento de Datos Faltantes

In [ ]:
print("="*80)
print("PREGUNTA 3: ESTRATEGIA DE TRATAMIENTO DE DATOS FALTANTES")
print("="*80)
print()
print("TOLERANCIA MÁXIMA: 25% de datos faltantes")
print()

# Análisis detallado
print("RECOMENDACIONES POR VARIABLE:")
print("-" * 80)

estrategia = {}

for idx, row in faltabilidad.iterrows():
    var = row['Variable']
    pct = row['Porcentaje_Faltante']
    
    if pct == 0:
        estrategia[var] = "SIN ACCIÓN - No hay datos faltantes"
        print(f"✓ {var}: {pct:.2f}%")
        print(f"  → Estrategia: SIN ACCIÓN")
        
    elif pct <= 25:
        # Variables numéricas vs categóricas
        if var in ['PM10', 'PM2.5', 'SO2', 'CO', 'NOX', 'NH3', 'CN', 'Número_de_empresas']:
            estrategia[var] = "IMPUTACIÓN - Mediana"
            print(f"✓ {var}: {pct:.2f}%")
            print(f"  → Estrategia: IMPUTACIÓN POR MEDIANA")
            print(f"  → Justificación: Datos numéricos, faltabilidad bajo tolerancia.")
            print(f"     La mediana es robusta ante outliers en variables de emisiones.")
        else:
            estrategia[var] = "SIN ACCIÓN - Variable categórica sin faltantes"
            print(f"✓ {var}: {pct:.2f}%")
            print(f"  → Estrategia: SIN ACCIÓN")
    else:
        estrategia[var] = "ELIMINAR VARIABLE"
        print(f"✗ {var}: {pct:.2f}% (SUPERA TOLERANCIA)")
        print(f"  → Estrategia: ELIMINAR VARIABLE")
        print(f"  → Justificación: Faltabilidad > 25%. Demasiada información pérdida.")
    print()

print("-" * 80)
print()
print("JUSTIFICACIÓN DE MÉTODOS SELECCIONADOS:")
print("-" * 80)
print()
print("1. IMPUTACIÓN POR MEDIANA (para variables de emisiones):")
print("   • La mediana es más robusta que la media ante outliers")
print("   • Las emisiones químicas frecuentemente presentan distribuciones asimétricas")
print("   • No introduce sesgo como lo haría la media con datos contaminados")
print("   • Mantiene la magnitud y orden de las observaciones")
print()
print("2. IMPUTACIÓN POR MEDIANA (para Número_de_empresas):")
print("   • Es una variable discreta (conteo) pero se puede usar mediana")
print("   • Alternativa: usar moda si datos muy discretos")
print()

---

## PREGUNTA 4: Implementación de Imputación

In [ ]:
print("="*80)
print("PREGUNTA 4: IMPUTACIÓN DE DATOS FALTANTES")
print("="*80)
print()

# Crear una copia del dataframe original
df_imputado = df.copy()

print("ANTES DE IMPUTACIÓN:")
print(f"Valores faltantes totales: {df_imputado.isnull().sum().sum()}")
print()

# Variables a imputar (aquellas con faltabilidad <= 25%)
variables_a_imputar = faltabilidad[faltabilidad['Porcentaje_Faltante'] <= 25]['Variable'].tolist()
variables_a_imputar = [var for var in variables_a_imputar if var not in ['Ciudad', 'País']]  # Excluir categóricas

print(f"Variables a imputar: {variables_a_imputar}")
print()

# Realizar imputación por mediana
for var in variables_a_imputar:
    if df_imputado[var].isnull().sum() > 0:
        mediana = df_imputado[var].median()
        print(f"\nImputando {var}:")
        print(f"  • Valores faltantes: {df_imputado[var].isnull().sum()}")
        print(f"  • Mediana calculada: {mediana:.4f}")
        df_imputado[var].fillna(mediana, inplace=True)
        print(f"  • ✓ Imputación completada")

print()
print("="*80)
print("DESPUÉS DE IMPUTACIÓN:")
print("="*80)
print(f"Valores faltantes totales: {df_imputado.isnull().sum().sum()}")
print()
print("Verificación de datos faltantes por variable:")
print(df_imputado.isnull().sum())

### Comparación: Antes vs Después

In [ ]:
# Tabla comparativa
print("\n" + "="*80)
print("COMPARACIÓN: ANTES VS DESPUÉS DE IMPUTACIÓN")
print("="*80)

comparacion = pd.DataFrame({
    'Variable': df.columns,
    'Faltantes_Antes': df.isnull().sum().values,
    'Faltantes_Después': df_imputado.isnull().sum().values,
    'Eliminados': (df.isnull().sum() - df_imputado.isnull().sum()).values
})

print(comparacion.to_string(index=False))
print()
print(f"Total faltantes antes: {df.isnull().sum().sum()}")
print(f"Total faltantes después: {df_imputado.isnull().sum().sum()}")
print(f"✓ Reducción exitosa: {df.isnull().sum().sum() - df_imputado.isnull().sum().sum()} valores imputados")

### Dataset después de imputación

In [ ]:
print("\n" + "="*80)
print("DATASET DESPUÉS DE IMPUTACIÓN (PRIMERAS 10 FILAS)")
print("="*80)
print()
df_imputado.head(10)

In [ ]:
print("\n" + "="*80)
print("ESTADÍSTICAS DESCRIPTIVAS DESPUÉS DE IMPUTACIÓN")
print("="*80)
print()
df_imputado.describe().round(4)

---

## PREGUNTA 5: Discusión sobre Imputación en Variables de Emisiones

In [ ]:
print("="*80)
print("PREGUNTA 5: CONSECUENCIAS Y CONSIDERACIONES ÉTICAS")
print("="*80)
print()

print("█" * 80)
print("PARTE A: CONSECUENCIAS DE IMPUTAR EN VARIABLES DE EMISIONES QUÍMICAS")
print("█" * 80)
print()

print("❌ RIESGOS POTENCIALES:")
print("-" * 80)
print()
print("1. ENMASCARAMIENTO DE LA REALIDAD:")
print("   • Las imputaciones suavizarán la distribución real de emisiones")
print("   • Datos faltantes podrían indicar: falta de monitoreo, negligencia regulatoria")
print("   • Al imputar, ocultamos estas señales de alarma")
print()

print("2. DISTORSIÓN DE ANÁLISIS DE RIESGO:")
print("   • La salud pública depende de mediciones PRECISAS de contaminación")
print("   • Una mediana imputada ≠ una medición real en el terreno")
print("   • Riesgo: autoridades toman decisiones basadas en datos artificiales")
print()

print("3. VARIABILIDAD NO CAPTURADA:")
print("   • Las emisiones varían significativamente según: estación, industrias activas")
print("   • Una mediana global pierde esta variabilidad temporal/espacial")
print("   • Puede subestimar o sobreestimar riesgos locales")
print()

print("4. RESPONSABILIDAD CORPORATIVA:")
print("   • Muchas ciudades tienen datos faltantes porque empresas NO REPORTAN")
print("   • Imputar 'oculta' esta falta de transparencia")
print("   • Debilita incentivos para que industria mida y reporte honestamente")
print()

print("5. SESGO EN DECISIONES DE POLÍTICA PÚBLICA:")
print("   • Si ciudades con malas prácticas tiene más datos faltantes")
print("   • Imputar medianas podría SUBESTIMAR su verdadera contaminación")
print("   • Resultado: menos inversión en control ambiental donde más se necesita")
print()

print()
print("█" * 80)
print("PARTE B: LÍNEA DE PENSAMIENTO RECOMENDADA PARA EQUIPOS DE DATA ANALYTICS")
print("█" * 80)
print()

print("🔍 PRINCIPIO FUNDAMENTAL: 'Datos Incompletos ≠ Datos Completables'")
print("-" * 80)
print()

print("1. INVESTIGAR ANTES DE IMPUTAR:")
print("   ✓ ¿POR QUÉ faltan los datos?")
print("     - Fallo técnico de sensores")
print("     - Falta de reporte de la empresa")
print("     - Cambio en métodos de medición")
print("   ✓ El mecanismo de falta importa MÁS que el porcentaje")
print()

print("2. CONTEXTO SECTORIAL:")
print("   ✓ Para datos ambiental/médico/financiero: tolerar > faltabilidad")
print("   ✓ Mejor: REPORTAR datos parciales que datos 'completados' falsos")
print("   ✓ Máxima transparencia: documentar qué fue imputado y cómo")
print()

print("3. DEFINIR TOLERANCIAS CON STAKEHOLDERS:")
print("   ✓ El 25% no es una regla universal, es un 'acuerdo de negocio'")
print("   ✓ Preguntar: ¿Qué nivel de precisión necesita tu decisión?")
print("   ✓ Ejemplo: Para política pública = 0-10% faltabilidad aceptable")
print("             Para análisis exploratorio = hasta 40% podría ser OK")
print()

print("4. ANÁLISIS DE SENSIBILIDAD:")
print("   ✓ No hacer SOLO un análisis con imputados")
print("   ✓ Hacer análisis MÚLTIPLES:")
print("     a) Eliminando registros con datos faltantes (caso pesimista)")
print("     b) Imputando medianas (caso neutral)")
print("     c) Imputando P75/P90 (caso optimista para validar límites)")
print("   ✓ Si conclusiones son iguales en 3 casos → robustas")
print("   ✓ Si conclusiones cambian → problema de datos faltantes, reportar")
print()

print("5. DOCUMENTACIÓN Y TRANSPARENCIA:")
print("   ✓ SIEMPRE reportar en anexo:")
print("     - Cuántos datos faltaban")
print("     - Qué método de imputación se usó")
print("     - Limitaciones de conclusiones por datos faltantes")
print("   ✓ Usar marcas/banderas en datos imputados (metadata)")
print("   ✓ Permitir auditoría y reproducibilidad")
print()

print("6. ÉTICA Y RESPONSABILIDAD:")
print("   ✓ En contextos de SALUD PÚBLICA:")
print("     - El dato faltante ES información (indica negligencia regulatoria)")
print("     - Mejor: Reportar incertidumbre que ocultar con imputaciones")
print("   ✓ Pregunta clave: ¿Mi análisis ENGAÑA o ESCLARECE?")
print()

print()
print("█" * 80)
print("RECOMENDACIÓN FINAL PARA ESTE CASO ESPECÍFICO")
print("█" * 80)
print()
print("Dado que estamos analizando CONTAMINACIÓN AMBIENTAL:")
print()
print("✅ OPCIÓN RECOMENDADA (ENFOQUE CONSERVADOR):")
print("-" * 80)
print("1. Imputar variables con < 10% faltabilidad (seguro)")
print("2. Crear variable DUMMY 'Dato_Faltante' = 1 para imputados")
print("3. En análisis, usar dos modelos:")
print("   a) Con imputación (para reportar tendencias)")
print("   b) Sin datos faltantes (para conclusiones críticas)")
print("4. En informe: DESTACAR cuáles ciudades tienen datos incompletos")
print("5. Recomendación: MEJORAR MONITOREO en ciudades con datos faltantes")
print()

---

## Resumen Visual: Análisis de Datos Faltantes

In [ ]:
# Gráfico de correlación entre variables numéricas después de imputación
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Matriz de correlación
ax1 = axes[0, 0]
numeric_cols = df_imputado.select_dtypes(include=[np.number]).columns
corr_matrix = df_imputado[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax1, cbar_kws={'label': 'Correlación'})
ax1.set_title('Matriz de Correlación - Emisiones Químicas (Después de Imputación)', fontsize=12, fontweight='bold')

# 2. Comparación Faltantes Antes/Después
ax2 = axes[0, 1]
x = np.arange(len(comparacion))
width = 0.35
ax2.bar(x - width/2, comparacion['Faltantes_Antes'], width, label='Antes', alpha=0.8, color='red')
ax2.bar(x + width/2, comparacion['Faltantes_Después'], width, label='Después', alpha=0.8, color='green')
ax2.set_ylabel('Número de Valores Faltantes', fontweight='bold')
ax2.set_title('Valores Faltantes: Antes vs Después de Imputación', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(comparacion['Variable'], rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. Distribuciones antes de imputación (ejemplo: PM2.5)
ax3 = axes[1, 0]
ax3.hist(df['PM2.5'].dropna(), bins=15, alpha=0.7, color='skyblue', edgecolor='black')
ax3.axvline(df['PM2.5'].median(), color='red', linestyle='--', linewidth=2, label=f'Mediana: {df["PM2.5"].median():.2f}')
ax3.set_xlabel('PM2.5 (μg/m³)', fontweight='bold')
ax3.set_ylabel('Frecuencia', fontweight='bold')
ax3.set_title('Distribución PM2.5 (antes de imputación)', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Distribuciones después de imputación
ax4 = axes[1, 1]
ax4.hist(df_imputado['PM2.5'], bins=15, alpha=0.7, color='lightgreen', edgecolor='black')
ax4.axvline(df_imputado['PM2.5'].median(), color='red', linestyle='--', linewidth=2, label=f'Mediana: {df_imputado["PM2.5"].median():.2f}')
ax4.set_xlabel('PM2.5 (μg/m³)', fontweight='bold')
ax4.set_ylabel('Frecuencia', fontweight='bold')
ax4.set_title('Distribución PM2.5 (después de imputación)', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Análisis visual completado")

---

## Exportar Dataset Imputado

In [ ]:
# Guardar el dataset imputado
output_path = 'Sesion_5_Contaminacion_Ciudades_Asiaticas_IMPUTADO.xlsx'
df_imputado.to_excel(output_path, index=False)
print(f"✓ Dataset imputado guardado en: {output_path}")
print(f"  Dimensión final: {df_imputado.shape}")
print(f"  Valores faltantes: {df_imputado.isnull().sum().sum()}")

---

## Conclusiones Finales

In [ ]:
print("\n" + "="*80)
print("CONCLUSIONES DEL ANÁLISIS")
print("="*80)
print()
print("1. DESCRIPCIÓN DE DATOS:")
print(f"   • Dataset contiene {df.shape[0]} ciudades asiáticas con {df.shape[1]} variables")
print(f"   • Variables: Ciudad, País, Número_de_empresas, y 6 contaminantes químicos")
print(f"   • Observación: Top 3 contaminantes por PM2.5 son {', '.join(df.nlargest(3, 'PM2.5')['Ciudad'].tolist())}")
print()

print("2. DATOS FALTANTES:")
faltantes_por_var = df.isnull().sum()
print(f"   • Total de valores faltantes: {df.isnull().sum().sum()} ({df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.2f}% del dataset)")
print(f"   • Variables afectadas: {(df.isnull().sum() > 0).sum()} de {df.shape[1]}")
print(f"   • Estrategia: Se imputaron variables con ≤25% faltabilidad por MEDIANA")
print()

print("3. IMPUTACIÓN REALIZADA:")
print(f"   ✓ Método: Imputación por MEDIANA (robusta ante outliers)")
print(f"   ✓ Valores imputados: {df.isnull().sum().sum()}")
print(f"   ✓ Variables imputadas: {', '.join(variables_a_imputar)}")
print()

print("4. CONSIDERACIONES ÉTICAS (Respuesta a P5):")
print("   • En contextos ambientales/salud: datos incompletos = información")
print("   • Riesgo: imputación puede ocultar negligencia regulatoria")
print("   • Recomendación: usar análisis de sensibilidad (con/sin imputación)")
print("   • Máxima transparencia: reportar qué fue imputado y cómo")
print()

print("5. PRÓXIMOS PASOS:")
print("   • Realizar análisis exploratorio completo con datos imputados")
print("   • Investigar outliers extremos en contaminación")
print("   • Correlacionar emisiones con número de empresas")
print("   • Comparar países y ciudades por índices de contaminación")
print("   • Crear recomendaciones de política ambiental")
print()
print("="*80)
print("✓ ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*80)